In [ ]:
# script to compare the correlations of predictions across each prediction type #
# ie panel one - are we identifying the same functional variants by effect size #
# updates from V1 (archived):
# tabix filtered predictions are now from the correct BED file

In [1]:
import pandas as pd
import matplotlib.pyplot as mpl
import seaborn as sns
from collections import Counter
from tqdm import tqdm
import torch
from scipy import stats
import numpy as np
import matplotlib


In [2]:
# define function to convert predictions to final df
def vcf2df (pred_df):
    # make lists of preds for new column
    # k562
    k_ref = []
    k_alt = []
    k_skew = []
    # hepg2
    h_ref = []
    h_alt = []
    h_skew = []
    # sknsh
    s_ref = []
    s_alt = []
    s_skew = []
    gene_name = []
    # parse predictions in 'INFO' column
    for i in tqdm(pred_df['INFO']):
        all_preds = i.split(';')
        # k562
        k_ref.append(float(all_preds[0].split('=')[-1]))
        k_alt.append(float(all_preds[3].split('=')[-1]))
        k_skew.append(float(all_preds[6].split('=')[-1]))
        # hepg2
        h_ref.append(float(all_preds[1].split('=')[-1]))
        h_alt.append(float(all_preds[4].split('=')[-1]))
        h_skew.append(float(all_preds[7].split('=')[-1]))
        # sknsh
        s_ref.append(float(all_preds[2].split('=')[-1]))
        s_alt.append(float(all_preds[5].split('=')[-1]))
        s_skew.append(float(all_preds[8].split('=')[-1]))
        # add annotated gene name
        gene_name.append(all_preds[-1].split('=')[-1])
    
    df = pd.DataFrame({'chrom' : pred_df['#CHROM'],
                       'pos' : pred_df['POS'],
                       'id' : pred_df['ID'],
                       'ref' : pred_df['REF'],
                       'alt' : pred_df['ALT'],
                       'k562_ref_pred' : k_ref,
                       'k562_alt_pred' : k_alt,
                       'k562_skew_pred' : k_skew,
                       'hepg2_ref_pred' : h_ref,
                       'hepg2_alt_pred' : h_alt,
                       'hepg2_skew_pred' : h_skew,
                       'sknsh_ref_pred' : s_ref,
                       'sknsh_alt_pred' : s_alt,
                       'sknsh_skew_pred' : s_skew,
                       'gene_name' : gene_name})
    return df

In [3]:
# define function to convert predictions to final df
def vcf2df_numCol (pred_df):
    # make lists of preds for new column
    # k562
    k_ref = []
    k_alt = []
    k_skew = []
    # hepg2
    h_ref = []
    h_alt = []
    h_skew = []
    # sknsh
    s_ref = []
    s_alt = []
    s_skew = []
    # parse predictions in 'INFO' column
    for i in tqdm(pred_df[7]):
        all_preds = i.split(';')
        # k562
        k_ref.append(float(all_preds[0].split('=')[-1]))
        k_alt.append(float(all_preds[3].split('=')[-1]))
        k_skew.append(float(all_preds[6].split('=')[-1]))
        # hepg2
        h_ref.append(float(all_preds[1].split('=')[-1]))
        h_alt.append(float(all_preds[4].split('=')[-1]))
        h_skew.append(float(all_preds[7].split('=')[-1]))
        # sknsh
        s_ref.append(float(all_preds[2].split('=')[-1]))
        s_alt.append(float(all_preds[5].split('=')[-1]))
        s_skew.append(float(all_preds[8].split('=')[-1]))
    
    df = pd.DataFrame({'chrom' : pred_df[0],
                       'pos' : pred_df[1],
                       'id' : pred_df[2],
                       'ref' : pred_df[3],
                       'alt' : pred_df[4],
                       'k562_ref_pred' : k_ref,
                       'k562_alt_pred' : k_alt,
                       'k562_skew_pred' : k_skew,
                       'hepg2_ref_pred' : h_ref,
                       'hepg2_alt_pred' : h_alt,
                       'hepg2_skew_pred' : h_skew,
                       'sknsh_ref_pred' : s_ref,
                       'sknsh_alt_pred' : s_alt,
                       'sknsh_skew_pred' : s_skew})
    return df

In [4]:
def read_in_sat_mut (path2satmut, chunksize):
    chunks2cat = []
    for chunk in tqdm(pd.read_csv(path2satmut, sep = '\t', chunksize=chunksize, header = None)):
        chunks2cat.append(chunk)
    cat_df = pd.concat(chunks2cat)
    return cat_df

In [5]:
### 093025 NEW BED CHECK ###
# open mpac predictions and reformat for downstream analyses
mpac_preds = vcf2df_numCol(read_in_sat_mut('../mpac/processed_data/all.mpac.preds.tabix.filtered.gencode.250bp.093025.vcf', 100000))
# calculate average skew for mpac predictions
mpac_preds.loc[:,'mean_skew_pred'] = [np.mean([k_skew, h_skew, s_skew]) for k_skew, h_skew, s_skew in zip(mpac_preds['k562_skew_pred'],
                                                                                                          mpac_preds['hepg2_skew_pred'],
                                                                                                          mpac_preds['sknsh_skew_pred'])]
# add gene name
mpac_preds.loc[:,'gene_name'] = [i.split('..')[0].split('_')[-1] for i in mpac_preds['id']]

155it [00:43,  3.60it/s]
100%|██████████| 15485844/15485844 [00:46<00:00, 336277.20it/s]


In [16]:
# filter for all mybpc3 predictions
mybpc3_preds = mpac_preds[(mpac_preds['gene_name'] == 'MYBPC3')]
# filter for only emVars
mybpc3_emvars = mybpc3_preds[mybpc3_preds['k562_skew_pred'] > 0.5]

In [17]:
mybpc3_emvars

,chrom,pos,id,ref,alt,k562_ref_pred,k562_alt_pred,k562_skew_pred,hepg2_ref_pred,hepg2_alt_pred,hepg2_skew_pred,sknsh_ref_pred,sknsh_alt_pred,sknsh_skew_pred,mean_skew_pred,gene_name
8601652,chr11,47352704,ENSG00000134571.12_ENST00000545968.6_MYBPC3..c...,A,G,1.447181,2.288417,0.841236,0.547128,0.911252,0.364124,0.090955,0.534787,0.443832,0.549731,MYBPC3
8601661,chr11,47352707,ENSG00000134571.12_ENST00000545968.6_MYBPC3..c...,G,C,1.404334,1.934531,0.530197,0.537967,0.839943,0.301976,0.058371,0.349595,0.291225,0.374466,MYBPC3
8601663,chr11,47352708,ENSG00000134571.12_ENST00000545968.6_MYBPC3..c...,G,A,1.441422,2.109063,0.667641,0.602694,0.929351,0.326657,0.072635,0.318747,0.246112,0.413470,MYBPC3
8601664,chr11,47352708,ENSG00000134571.12_ENST00000545968.6_MYBPC3..c...,G,C,1.441422,2.159357,0.717935,0.602694,0.986260,0.383567,0.072635,0.374049,0.301415,0.467639,MYBPC3
8601665,chr11,47352708,ENSG00000134571.12_ENST00000545968.6_MYBPC3..c...,G,T,1.441422,1.959310,0.517888,0.602694,0.781177,0.178483,0.072635,0.254246,0.181612,0.292661,MYBPC3
8601666,chr11,47352709,ENSG00000134571.12_ENST00000545968.6_MYBPC3..c...,G,A,1.481896,2.281719,0.799822,0.601327,1.020508,0.419181,0.091104,0.485306,0.394202,0.537735,MYBPC3
8601667,chr11,47352709,ENSG00000134571.12_ENST00000545968.6_MYBPC3..c...,G,C,1.481896,2.399010,0.917114,0.601327,1.125655,0.524328,0.091104,0.530945,0.439841,0.627094,MYBPC3
8601668,chr11,47352709,ENSG00000134571.12_ENST00000545968.6_MYBPC3..c...,G,T,1.481896,2.201273,0.719376,0.601327,0.968614,0.367287,0.091104,0.329400,0.238295,0.441653,MYBPC3
8601671,chr11,47352710,ENSG00000134571.12_ENST00000545968.6_MYBPC3..c...,G,T,1.531184,2.065382,0.534198,0.604164,0.945653,0.341489,0.079646,0.399125,0.319479,0.398389,MYBPC3
8601673,chr11,47352711,ENSG00000134571.12_ENST00000545968.6_MYBPC3..c...,T,C,1.492760,2.180345,0.687584,0.614459,0.992340,0.377881,0.109389,0.370523,0.261135,0.442200,MYBPC3


In [18]:
len(mybpc3_emvars['pos'].unique())

18

In [14]:
len(mybpc3_emvars)

70

In [15]:
mybpc3_emvars[mybpc3_emvars['k562_skew_pred'] < 0]

,chrom,pos,id,ref,alt,k562_ref_pred,k562_alt_pred,k562_skew_pred,hepg2_ref_pred,hepg2_alt_pred,hepg2_skew_pred,sknsh_ref_pred,sknsh_alt_pred,sknsh_skew_pred,mean_skew_pred,gene_name
8601752,chr11,47352737,ENSG00000134571.12_ENST00000545968.6_MYBPC3..c...,A,T,1.812192,1.268642,-0.543549,0.717516,0.537751,-0.179765,0.132239,0.008993,-0.123246,-0.282187,MYBPC3
8601755,chr11,47352738,ENSG00000134571.12_ENST00000545968.6_MYBPC3..c...,C,T,1.884968,1.349974,-0.534995,0.785281,0.521189,-0.264091,0.150361,0.054551,-0.095810,-0.298299,MYBPC3
8601756,chr11,47352739,ENSG00000134571.12_ENST00000545968.6_MYBPC3..c...,C,A,1.872555,1.317637,-0.554919,0.741293,0.508389,-0.232905,0.123131,-0.020777,-0.143908,-0.310577,MYBPC3
8601759,chr11,47352740,ENSG00000134571.12_ENST00000545968.6_MYBPC3..c...,T,A,1.912450,1.262475,-0.649975,0.757063,0.447697,-0.309366,0.137035,-0.038488,-0.175523,-0.378288,MYBPC3
8601760,chr11,47352740,ENSG00000134571.12_ENST00000545968.6_MYBPC3..c...,T,C,1.912450,1.023986,-0.888464,0.757063,0.441507,-0.315555,0.137035,-0.072800,-0.209835,-0.471285,MYBPC3
8601767,chr11,47352742,ENSG00000134571.12_ENST00000545968.6_MYBPC3..c...,C,T,1.939469,1.364135,-0.575334,0.850375,0.530718,-0.319657,0.250214,0.002859,-0.247354,-0.380782,MYBPC3
8601768,chr11,47352743,ENSG00000134571.12_ENST00000545968.6_MYBPC3..c...,C,A,2.004600,1.243631,-0.760969,0.832999,0.464939,-0.368059,0.229005,-0.047339,-0.276344,-0.468458,MYBPC3
8601774,chr11,47352745,ENSG00000134571.12_ENST00000545968.6_MYBPC3..c...,C,A,2.056597,1.547487,-0.509109,0.906032,0.641296,-0.264737,0.258591,0.090734,-0.167857,-0.313901,MYBPC3
8601775,chr11,47352745,ENSG00000134571.12_ENST00000545968.6_MYBPC3..c...,C,G,2.056597,1.507726,-0.548871,0.906032,0.671413,-0.234619,0.258591,0.112196,-0.146395,-0.309962,MYBPC3
8601783,chr11,47352748,ENSG00000134571.12_ENST00000545968.6_MYBPC3..c...,C,A,1.980760,1.451963,-0.528797,0.799048,0.678300,-0.120749,0.131732,0.057594,-0.074138,-0.241228,MYBPC3
